# 📊 Análise Histórica de Traders — Polymarket

**Objetivo:** Analisar o comportamento histórico dos top traders do Polymarket:
- Evolução de ranking ao longo do tempo
- Estratégias de investimento na linha do tempo
- Padrões de entrada/saída de mercados
- Concentração vs diversificação
- Análise de PnL acumulado

**Dados:** 788 traders, 1.6M+ trades, 32K posições ativas

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

# Ajustar para o caminho correto no seu ambiente
TRADERS_DIR = 'historical/traders/'

# Se estiver rodando no MacBook, descomentar:
# TRADERS_DIR = '/Users/possebon/polymarket_data/traders/'

print(f'Carregando traders de: {TRADERS_DIR}')
print(f'Arquivos encontrados: {len(os.listdir(TRADERS_DIR))}')

## 1. Carregamento e Preparação dos Dados

In [ ]:
# Carregar todos os traders
traders_data = []
all_trades = []
all_positions = []

for filename in os.listdir(TRADERS_DIR):
    if not filename.endswith('.json'):
        continue
    with open(os.path.join(TRADERS_DIR, filename)) as f:
        data = json.load(f)
    
    info = data.get('info', {})
    profile = data.get('profile', {})
    username = info.get('username', 'unknown')
    wallet = data.get('wallet', '')
    
    traders_data.append({
        'wallet': wallet,
        'username': username,
        'pnl': info.get('pnl', 0),
        'volume': info.get('volume', 0),
        'markets_traded': info.get('markets_traded', 0),
        'categories': info.get('categories', []),
        'ranks': info.get('ranks', {}),
        'trades_count': data.get('trades_count', 0),
        'positions_count': data.get('positions_count', 0),
        'created_at': profile.get('createdAt', ''),
    })
    
    # Trades
    for t in data.get('trades', []):
        all_trades.append({
            'wallet': wallet,
            'username': username,
            'timestamp': t.get('timestamp', 0),
            'type': t.get('type', ''),
            'side': t.get('side', ''),
            'size': t.get('size', 0),
            'usdc_size': t.get('usdcSize', 0),
            'price': t.get('price', 0),
            'title': t.get('title', ''),
            'slug': t.get('slug', ''),
            'outcome': t.get('outcome', ''),
            'outcome_index': t.get('outcomeIndex', 0),
        })
    
    # Positions
    for p in data.get('positions', []):
        all_positions.append({
            'wallet': wallet,
            'username': username,
            'title': p.get('title', ''),
            'slug': p.get('slug', ''),
            'size': p.get('size', 0),
            'avg_price': p.get('avgPrice', 0),
            'initial_value': p.get('initialValue', 0),
            'current_value': p.get('currentValue', 0),
            'cash_pnl': p.get('cashPnl', 0),
            'percent_pnl': p.get('percentPnl', 0),
            'realized_pnl': p.get('realizedPnl', 0),
            'cur_price': p.get('curPrice', 0),
            'outcome': p.get('outcome', ''),
            'end_date': p.get('endDate', ''),
        })

df_traders = pd.DataFrame(traders_data)
df_trades = pd.DataFrame(all_trades)
df_positions = pd.DataFrame(all_positions)

# Converter timestamps
df_trades['datetime'] = pd.to_datetime(df_trades['timestamp'], unit='s', utc=True)
df_trades['date'] = df_trades['datetime'].dt.date
df_trades['month'] = df_trades['datetime'].dt.to_period('M')
df_trades['week'] = df_trades['datetime'].dt.to_period('W')

# Filtrar só trades reais (não YIELD, REWARD, etc)
df_real_trades = df_trades[df_trades['type'] == 'TRADE'].copy()

print(f'✅ {len(df_traders)} traders carregados')
print(f'✅ {len(df_trades):,} trades totais')
print(f'   - TRADE: {len(df_real_trades):,}')
print(f'   - Outros: {len(df_trades) - len(df_real_trades):,} (YIELD, MERGE, REDEEM, REWARD)')
print(f'✅ {len(df_positions):,} posições ativas')
print(f'\nPeríodo: {df_trades["datetime"].min()} → {df_trades["datetime"].max()}')

## 2. Overview dos Traders

In [ ]:
# Top 20 por PnL
top20 = df_traders.nlargest(20, 'pnl')[['username', 'pnl', 'trades_count', 'positions_count']].copy()
top20['pnl_fmt'] = top20['pnl'].apply(lambda x: f'${x:,.0f}')
top20 = top20.reset_index(drop=True)
top20.index += 1
top20.index.name = 'rank'
print('🏆 TOP 20 TRADERS POR PnL\n')
print(top20[['username', 'pnl_fmt', 'trades_count', 'positions_count']].to_string())

In [ ]:
# Distribuição de PnL
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PnL distribution (log scale)
ax = axes[0]
pnl_pos = df_traders[df_traders['pnl'] > 0]['pnl']
ax.hist(pnl_pos, bins=50, color='#2ecc71', alpha=0.7, edgecolor='white')
ax.set_xscale('log')
ax.set_title('Distribuição de PnL (positivos)', fontsize=13, fontweight='bold')
ax.set_xlabel('PnL (USD, escala log)')
ax.set_ylabel('Traders')

# Trades por trader
ax = axes[1]
trades_per = df_real_trades.groupby('username').size()
ax.hist(trades_per, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
ax.set_title('Trades por Trader', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de trades')
ax.set_ylabel('Traders')

# Posições ativas por trader
ax = axes[2]
ax.hist(df_traders['positions_count'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='white')
ax.set_title('Posições Ativas por Trader', fontsize=13, fontweight='bold')
ax.set_xlabel('Posições')
ax.set_ylabel('Traders')

plt.tight_layout()
plt.show()

## 3. Evolução Temporal — Volume e Atividade

In [ ]:
# Volume diário e número de trades ao longo do tempo
daily = df_real_trades.groupby('date').agg(
    volume=('usdc_size', 'sum'),
    num_trades=('usdc_size', 'count'),
    unique_traders=('username', 'nunique'),
    unique_markets=('slug', 'nunique'),
).reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Volume
ax = axes[0]
ax.fill_between(daily['date'], daily['volume'], alpha=0.4, color='#2ecc71')
ax.plot(daily['date'], daily['volume'].rolling(7).mean(), color='#27ae60', linewidth=2, label='Média 7d')
ax.set_title('Volume Diário (USDC)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
ax.legend()
ax.grid(alpha=0.3)

# Trades
ax = axes[1]
ax.bar(daily['date'], daily['num_trades'], alpha=0.5, color='#3498db', width=1)
ax.plot(daily['date'], daily['num_trades'].rolling(7).mean(), color='#2980b9', linewidth=2, label='Média 7d')
ax.set_title('Número de Trades por Dia', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# Traders ativos
ax = axes[2]
ax.plot(daily['date'], daily['unique_traders'], color='#e74c3c', linewidth=1.5)
ax.plot(daily['date'], daily['unique_traders'].rolling(7).mean(), color='#c0392b', linewidth=2, label='Média 7d')
ax.set_title('Traders Ativos por Dia (do nosso universo)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Ranking Evolution — Top Traders ao Longo do Tempo

Vamos calcular o PnL acumulado de cada trader semana a semana para ver como o ranking evolui.

In [ ]:
# Calcular PnL aproximado por trade
# Para TRADE: se side=BUY, PnL negativo (gastou USDC); se side=SELL, PnL positivo
# Para REDEEM: PnL positivo (resgatou)
# Simplificação: usamos usdc_size com sinal

df_pnl = df_trades.copy()

# Atribuir fluxo de caixa
def calc_cashflow(row):
    if row['type'] == 'TRADE':
        if row['side'] == 'BUY':
            return -row['usdc_size']  # Pagou
        elif row['side'] == 'SELL':
            return row['usdc_size']   # Recebeu
    elif row['type'] == 'REDEEM':
        return row['usdc_size']  # Resgatou
    elif row['type'] == 'MERGE':
        return row['usdc_size']  # Merge = recebeu
    return 0

df_pnl['cashflow'] = df_pnl.apply(calc_cashflow, axis=1)

# PnL acumulado semanal por trader
weekly_pnl = df_pnl.groupby(['username', 'week']).agg(
    cashflow=('cashflow', 'sum'),
    num_trades=('cashflow', 'count'),
    volume=('usdc_size', 'sum'),
).reset_index()

# Calcular PnL acumulado
weekly_pnl = weekly_pnl.sort_values(['username', 'week'])
weekly_pnl['cum_pnl'] = weekly_pnl.groupby('username')['cashflow'].cumsum()

print(f'Semanas com dados: {weekly_pnl["week"].nunique()}')
print(f'Traders com trades: {weekly_pnl["username"].nunique()}')

In [ ]:
# Top 10 traders - Evolução do PnL Acumulado
top10_names = df_traders.nlargest(10, 'pnl')['username'].tolist()

fig, ax = plt.subplots(figsize=(16, 8))

colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i, name in enumerate(top10_names):
    trader_data = weekly_pnl[weekly_pnl['username'] == name].copy()
    trader_data['week_start'] = trader_data['week'].apply(lambda x: x.start_time)
    ax.plot(trader_data['week_start'], trader_data['cum_pnl'], 
            label=name, linewidth=2, color=colors[i])

ax.set_title('🏆 Top 10 Traders — PnL Acumulado (Cashflow)', fontsize=16, fontweight='bold')
ax.set_xlabel('Semana')
ax.set_ylabel('PnL Acumulado (USDC)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M' if abs(x) >= 1e6 else f'${x/1e3:.0f}K'))
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ranking semanal — quem sobe e quem desce
# Calcular rank por semana baseado no PnL acumulado
ranking_weekly = weekly_pnl.copy()
ranking_weekly['rank'] = ranking_weekly.groupby('week')['cum_pnl'].rank(ascending=False, method='min')

# Plotar evolução de ranking dos Top 15
top15_names = df_traders.nlargest(15, 'pnl')['username'].tolist()

fig, ax = plt.subplots(figsize=(16, 8))

colors = plt.cm.tab20(np.linspace(0, 1, 15))
for i, name in enumerate(top15_names):
    trader_rank = ranking_weekly[ranking_weekly['username'] == name].copy()
    trader_rank['week_start'] = trader_rank['week'].apply(lambda x: x.start_time)
    ax.plot(trader_rank['week_start'], trader_rank['rank'], 
            label=name, linewidth=2, color=colors[i], marker='o', markersize=3)

ax.invert_yaxis()  # Rank 1 no topo
ax.set_title('📈 Evolução de Ranking — Top 15 Traders', fontsize=16, fontweight='bold')
ax.set_xlabel('Semana')
ax.set_ylabel('Ranking (1 = melhor)')
ax.set_ylim(bottom=50, top=0)  # Limitar visualização
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Estratégias de Investimento — Análise por Mercado

In [ ]:
# Mercados mais populares entre os top traders
top50_names = df_traders.nlargest(50, 'pnl')['username'].tolist()
top50_trades = df_real_trades[df_real_trades['username'].isin(top50_names)]

market_pop = top50_trades.groupby('title').agg(
    volume=('usdc_size', 'sum'),
    trades=('usdc_size', 'count'),
    traders=('username', 'nunique'),
).sort_values('volume', ascending=False).head(20)

market_pop['volume_fmt'] = market_pop['volume'].apply(lambda x: f'${x:,.0f}')
print('🎯 TOP 20 MERCADOS POR VOLUME (Top 50 traders)\n')
print(market_pop[['volume_fmt', 'trades', 'traders']].to_string())

In [ ]:
# Timeline de mercados — quando os top traders entram e saem
# Pegar os 10 mercados mais negociados
top_markets = top50_trades.groupby('slug')['usdc_size'].sum().nlargest(10).index.tolist()
top_market_titles = top50_trades[top50_trades['slug'].isin(top_markets)].groupby('slug')['title'].first().to_dict()

fig, ax = plt.subplots(figsize=(16, 8))

for i, slug in enumerate(top_markets):
    market_trades = top50_trades[top50_trades['slug'] == slug]
    daily_vol = market_trades.groupby('date')['usdc_size'].sum()
    daily_vol.index = pd.to_datetime(daily_vol.index)
    title = top_market_titles.get(slug, slug)[:40]
    ax.fill_between(daily_vol.index, i, i + daily_vol / daily_vol.max() * 0.8, 
                    alpha=0.6, label=title)

ax.set_yticks(range(len(top_markets)))
ax.set_yticklabels([top_market_titles.get(s, s)[:45] for s in top_markets], fontsize=9)
ax.set_title('📅 Timeline de Mercados — Atividade dos Top 50 Traders', fontsize=14, fontweight='bold')
ax.set_xlabel('Data')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Perfil de Estratégia por Trader

Classificar traders por estilo:
- **Concentrado** vs **Diversificado** (quantos mercados)
- **Alta frequência** vs **Baixa frequência** (trades/dia)
- **Contrarian** vs **Momentum** (compra a preços baixos ou altos)

In [ ]:
# Calcular métricas de estilo por trader
trader_style = df_real_trades.groupby('username').agg(
    total_trades=('usdc_size', 'count'),
    total_volume=('usdc_size', 'sum'),
    unique_markets=('slug', 'nunique'),
    avg_price_bought=('price', lambda x: x[df_real_trades.loc[x.index, 'side'] == 'BUY'].mean()),
    median_trade_size=('usdc_size', 'median'),
    first_trade=('datetime', 'min'),
    last_trade=('datetime', 'max'),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
).reset_index()

# Dias ativos
trader_style['active_days'] = (trader_style['last_trade'] - trader_style['first_trade']).dt.days + 1
trader_style['trades_per_day'] = trader_style['total_trades'] / trader_style['active_days'].clip(lower=1)
trader_style['vol_per_market'] = trader_style['total_volume'] / trader_style['unique_markets'].clip(lower=1)
trader_style['buy_ratio'] = trader_style['buy_count'] / trader_style['total_trades']

# Merge com PnL
trader_style = trader_style.merge(df_traders[['username', 'pnl']], on='username', how='left')

# Classificação
trader_style['diversification'] = pd.qcut(trader_style['unique_markets'], 3, labels=['Concentrado', 'Moderado', 'Diversificado'])
trader_style['frequency'] = pd.qcut(trader_style['trades_per_day'], 3, labels=['Low-Freq', 'Mid-Freq', 'High-Freq'])

print('📊 PERFIS DE ESTRATÉGIA\n')
print('Diversificação:')
print(trader_style['diversification'].value_counts().to_string())
print('\nFrequência:')
print(trader_style['frequency'].value_counts().to_string())

In [ ]:
# Scatter: Diversificação vs Frequência vs PnL
fig, ax = plt.subplots(figsize=(14, 8))

# Filtrar traders com PnL positivo para log scale
plot_data = trader_style[trader_style['pnl'] > 100].copy()

scatter = ax.scatter(
    plot_data['unique_markets'],
    plot_data['trades_per_day'],
    c=np.log10(plot_data['pnl']),
    s=np.clip(plot_data['pnl'] / 1000, 10, 500),
    cmap='RdYlGn',
    alpha=0.6,
    edgecolors='white',
    linewidth=0.5,
)

# Anotar top 10
for _, row in plot_data.nlargest(10, 'pnl').iterrows():
    ax.annotate(row['username'], (row['unique_markets'], row['trades_per_day']),
               fontsize=8, fontweight='bold', ha='left',
               xytext=(5, 5), textcoords='offset points')

cbar = plt.colorbar(scatter, ax=ax, label='log10(PnL)')
ax.set_xlabel('Mercados Únicos (diversificação)', fontsize=12)
ax.set_ylabel('Trades por Dia (frequência)', fontsize=12)
ax.set_title('🎯 Mapa de Estratégias — Diversificação × Frequência × PnL', fontsize=14, fontweight='bold')
ax.grid(alpha=0.2)
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 7. Análise de Posições Atuais — Onde o Smart Money Está

In [ ]:
# Posições dos Top 30 traders
top30_names = df_traders.nlargest(30, 'pnl')['username'].tolist()
top_positions = df_positions[df_positions['username'].isin(top30_names)].copy()

# Mercados com mais capital dos top traders
market_capital = top_positions.groupby('title').agg(
    total_value=('current_value', 'sum'),
    total_pnl=('cash_pnl', 'sum'),
    avg_pnl_pct=('percent_pnl', 'mean'),
    traders=('username', 'nunique'),
    trader_list=('username', lambda x: ', '.join(sorted(x.unique()))),
).sort_values('total_value', ascending=False).head(20)

market_capital['value_fmt'] = market_capital['total_value'].apply(lambda x: f'${x:,.0f}')
market_capital['pnl_fmt'] = market_capital['total_pnl'].apply(lambda x: f'${x:,.0f}')

print('💰 ONDE ESTÁ O SMART MONEY — Posições dos Top 30 Traders\n')
for _, row in market_capital.iterrows():
    print(f'📌 {row.name[:60]}')
    print(f'   Capital: {row["value_fmt"]} | PnL: {row["pnl_fmt"]} ({row["avg_pnl_pct"]:.1f}%) | {row["traders"]} traders')
    print(f'   Traders: {row["trader_list"][:80]}')
    print()

In [ ]:
# Heatmap: Top 15 traders × Top 15 mercados
top15_names = df_traders.nlargest(15, 'pnl')['username'].tolist()
top15_markets = top_positions.groupby('title')['current_value'].sum().nlargest(15).index.tolist()

heatmap_data = top_positions[
    (top_positions['username'].isin(top15_names)) & 
    (top_positions['title'].isin(top15_markets))
].pivot_table(
    index='username', columns='title', values='current_value', aggfunc='sum'
).fillna(0)

# Reordenar
heatmap_data = heatmap_data.loc[
    [n for n in top15_names if n in heatmap_data.index],
    [m for m in top15_markets if m in heatmap_data.columns]
]

# Truncar nomes de mercado
heatmap_data.columns = [c[:35] + '...' if len(c) > 35 else c for c in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(16, 10))
im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)
ax.set_title('🔥 Heatmap: Capital dos Top Traders por Mercado', fontsize=14, fontweight='bold')

# Adicionar valores
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        if val > 0:
            ax.text(j, i, f'${val/1e3:.0f}K' if val >= 1000 else f'${val:.0f}',
                   ha='center', va='center', fontsize=7,
                   color='white' if val > heatmap_data.values.max() * 0.5 else 'black')

plt.colorbar(im, ax=ax, label='Capital (USDC)')
plt.tight_layout()
plt.show()

## 8. Padrões Temporais — Quando os Traders Operam

In [ ]:
# Hora do dia e dia da semana
df_real_trades['hour'] = df_real_trades['datetime'].dt.hour
df_real_trades['dayofweek'] = df_real_trades['datetime'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por hora
ax = axes[0]
hourly = df_real_trades.groupby('hour')['usdc_size'].agg(['sum', 'count'])
ax.bar(hourly.index, hourly['sum'] / 1e6, color='#3498db', alpha=0.7)
ax.set_xlabel('Hora (UTC)')
ax.set_ylabel('Volume ($M)')
ax.set_title('Volume por Hora do Dia (UTC)', fontsize=13, fontweight='bold')
ax.set_xticks(range(24))
ax.grid(alpha=0.3)

# Por dia da semana
ax = axes[1]
days_labels = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
dow = df_real_trades.groupby('dayofweek')['usdc_size'].agg(['sum', 'count'])
ax.bar(dow.index, dow['sum'] / 1e6, color='#e74c3c', alpha=0.7)
ax.set_xlabel('Dia da Semana')
ax.set_ylabel('Volume ($M)')
ax.set_title('Volume por Dia da Semana', fontsize=13, fontweight='bold')
ax.set_xticks(range(7))
ax.set_xticklabels(days_labels)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Deep Dive — Trader Individual

Escolha um trader para analisar em detalhe.

In [ ]:
# ========================================
# ALTERE AQUI para analisar outro trader
TARGET_TRADER = 'Theo4'  
# ========================================

t_trades = df_real_trades[df_real_trades['username'] == TARGET_TRADER].copy().sort_values('datetime')
t_positions = df_positions[df_positions['username'] == TARGET_TRADER].copy()
t_info = df_traders[df_traders['username'] == TARGET_TRADER].iloc[0]

print(f'═══════════════════════════════════════')
print(f'  🔍 DEEP DIVE: {TARGET_TRADER}')
print(f'═══════════════════════════════════════')
print(f'  PnL Total:      ${t_info["pnl"]:,.0f}')
print(f'  Trades:          {len(t_trades):,}')
print(f'  Posições ativas: {len(t_positions)}')
print(f'  Mercados únicos: {t_trades["slug"].nunique()}')
print(f'  Período:         {t_trades["datetime"].min().strftime("%Y-%m-%d")} → {t_trades["datetime"].max().strftime("%Y-%m-%d")}')
print(f'  Dias ativos:     {(t_trades["datetime"].max() - t_trades["datetime"].min()).days}')
print(f'  Volume total:    ${t_trades["usdc_size"].sum():,.0f}')
print(f'  Trade médio:     ${t_trades["usdc_size"].median():,.0f} (mediana)')
print(f'═══════════════════════════════════════')

In [ ]:
# Timeline do trader
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Volume diário
ax = axes[0]
daily_t = t_trades.groupby('date').agg(
    volume=('usdc_size', 'sum'),
    trades=('usdc_size', 'count'),
).reset_index()
daily_t['date'] = pd.to_datetime(daily_t['date'])
ax.bar(daily_t['date'], daily_t['volume'], color='#2ecc71', alpha=0.6, width=1)
ax.set_title(f'{TARGET_TRADER} — Volume Diário', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
ax.grid(alpha=0.3)

# Número de trades por dia
ax = axes[1]
ax.bar(daily_t['date'], daily_t['trades'], color='#3498db', alpha=0.6, width=1)
ax.set_title(f'{TARGET_TRADER} — Trades por Dia', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)

# Mercados ao longo do tempo
ax = axes[2]
top5_markets = t_trades.groupby('slug')['usdc_size'].sum().nlargest(5).index
for slug in top5_markets:
    m_trades = t_trades[t_trades['slug'] == slug]
    m_daily = m_trades.groupby('date')['usdc_size'].sum()
    m_daily.index = pd.to_datetime(m_daily.index)
    title = m_trades['title'].iloc[0][:35]
    ax.scatter(m_daily.index, m_daily.values, label=title, alpha=0.7, s=30)

ax.set_title(f'{TARGET_TRADER} — Top 5 Mercados (Volume por Dia)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e3:.0f}K'))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Posições atuais do trader
if len(t_positions) > 0:
    pos_sorted = t_positions.sort_values('current_value', ascending=False).head(15)
    print(f'\n💼 POSIÇÕES ATUAIS — {TARGET_TRADER}\n')
    for _, p in pos_sorted.iterrows():
        emoji = '🟢' if p['cash_pnl'] > 0 else '🔴'
        print(f'{emoji} {p["title"][:55]}')
        print(f'   Valor: ${p["current_value"]:,.0f} | PnL: ${p["cash_pnl"]:,.0f} ({p["percent_pnl"]:.1f}%) | Outcome: {p["outcome"]}')
        print()
else:
    print(f'{TARGET_TRADER} não tem posições ativas.')

## 10. Resumo Executivo

In [ ]:
print('═' * 60)
print('  📋 RESUMO EXECUTIVO — Análise Histórica Polymarket')
print('═' * 60)
print(f'''
📊 DADOS
  • {len(df_traders)} traders analisados
  • {len(df_real_trades):,} trades reais (TRADE)
  • {len(df_positions):,} posições ativas
  • Volume total: ${df_real_trades["usdc_size"].sum():,.0f}
  • Período: {df_real_trades["datetime"].min().strftime("%Y-%m-%d")} → {df_real_trades["datetime"].max().strftime("%Y-%m-%d")}

💰 PnL
  • Top 1 (Theo4): ${df_traders["pnl"].max():,.0f}
  • Mediana: ${df_traders["pnl"].median():,.0f}
  • PnL total (positivos): ${df_traders[df_traders["pnl"] > 0]["pnl"].sum():,.0f}

🎯 ESTRATÉGIAS OBSERVADAS
  • Mercados mais populares: {df_real_trades["slug"].value_counts().index[0][:40]}...
  • Traders concentrados vs diversificados: {(trader_style["diversification"] == "Concentrado").sum()} vs {(trader_style["diversification"] == "Diversificado").sum()}
  • Traders alta-freq vs baixa-freq: {(trader_style["frequency"] == "High-Freq").sum()} vs {(trader_style["frequency"] == "Low-Freq").sum()}
''')
print('═' * 60)